## Common Functions : Needed for Tasks 1, 2, 3 and 5

In [ ]:
import random
import math


def generate_initial_state(n=4):
    return [random.randrange(n) for _ in range(n)]


def calculate_heuristic(state):
    h = 0
    for i in range(len(state)):
        for j in range(i + 1, len(state)):
            if state[i] == state[j] or abs(state[i] - state[j]) == j - i:
                h += 1
    return h


def generate_neighbors(state):
    neighbors = []
    for col in range(len(state)):
        for row in range(len(state)):
            if row != state[col]:
                neighbors.append(state[:col] + [row] + state[col + 1:])
    return neighbors


def print_board(state):
    n = len(state)
    for row in range(n):
        print(" ".join("Q" if state[col] == row else "-" for col in range(n)))

### Task 1: Implement Hill Climbing for N-Queens

In [ ]:
def hill_climbing(initial_state, verbose=True):
    current = initial_state
    iterations = 0

    while True:
        current_h = calculate_heuristic(current)
        if verbose:
            print(f"Iteration {iterations}: State = {current}, h = {current_h}")

        if current_h == 0:
            if verbose:
                print("Solution found (h=0)")
            return current, iterations

        neighbors = generate_neighbors(current)
        best_neighbor = min(neighbors, key=calculate_heuristic)
        best_h = calculate_heuristic(best_neighbor)
        if verbose:
            print(f"    Best Neighbor = {best_neighbor}, h = {best_h}")

        if best_h >= current_h:
            if verbose:
                print("Stuck at local optimum")
            return current, iterations

        current = best_neighbor
        iterations += 1


initial_state = generate_initial_state()

solution, _ = hill_climbing(initial_state)

print("Final Board:")
print_board(solution)
print("Final h:", calculate_heuristic(solution))

Iteration 0: State = [0, 2, 0, 1], h = 2
    Best Neighbor = [0, 2, 3, 1], h = 1
Iteration 1: State = [0, 2, 3, 1], h = 1
    Best Neighbor = [0, 0, 3, 1], h = 1
Stuck at local optimum
Final Board:
Q - - -
- - - Q
- Q - -
- - Q -
Final h: 1


### Task 2: Random-Restart Hill Climbing

In [ ]:
def random_restart_hill_climbing(max_restarts=100, verbose=False):
    total_iterations = 0
    first_state = None

    for restart in range(max_restarts):
        initial_state = generate_initial_state()
        if first_state is None:
            first_state = initial_state

        solution, iters = hill_climbing(initial_state, verbose=False)
        total_iterations += iters
        if verbose:
            print(f"Restart {restart + 1}: start = {initial_state}, iterations = {iters}, h = {calculate_heuristic(solution)}")

        if calculate_heuristic(solution) == 0:
            return solution, restart + 1, total_iterations, first_state

    return None, max_restarts, total_iterations, first_state


solution, restarts, iters, first = random_restart_hill_climbing(verbose=True)

print("Solution:", solution)
print("Restarts:", restarts)
print_board(solution)

print("\nExperiment: 10 runs")
print(f"{'Run':<5}{'Initial State':<18}{'Restarts':<10}{'Total Iterations':<18}{'Final h'}")
restart_counts = []
for run in range(1, 11):
    solution, restarts, iters, first = random_restart_hill_climbing()
    restart_counts.append(restarts)
    print(f"{run:<5}{str(first):<18}{restarts:<10}{iters:<18}{calculate_heuristic(solution)}")
print("Average restarts:", sum(restart_counts) / len(restart_counts))

trials = 1000
single_success = sum(calculate_heuristic(hill_climbing(generate_initial_state(), verbose=False)[0]) == 0 for _ in range(trials))
rr_success = sum(random_restart_hill_climbing()[0] is not None for _ in range(trials))
print(f"\nSingle-start hill climbing success rate: {single_success / trials:.1%}")
print(f"Random-restart hill climbing success rate: {rr_success / trials:.1%}")

Restart 1: start = [1, 1, 0, 3], iterations = 1, h = 1
Restart 2: start = [2, 3, 2, 3], iterations = 2, h = 1
Restart 3: start = [3, 1, 0, 3], iterations = 1, h = 1
Restart 4: start = [0, 0, 2, 1], iterations = 1, h = 1
Restart 5: start = [1, 1, 0, 2], iterations = 1, h = 0
Solution: [1, 3, 0, 2]
Restarts: 5
- - Q -
Q - - -
- - - Q
- Q - -

Experiment: 10 runs
Run  Initial State     Restarts  Total Iterations  Final h
1    [1, 2, 1, 2]      1         2                 0
2    [0, 2, 0, 0]      3         5                 0
3    [1, 1, 1, 1]      4         8                 0
4    [0, 1, 2, 0]      3         4                 0
5    [0, 0, 1, 1]      1         2                 0
6    [3, 0, 2, 3]      4         5                 0
7    [3, 3, 3, 0]      2         3                 0
8    [2, 0, 0, 2]      5         9                 0
9    [2, 0, 3, 0]      1         1                 0
10   [0, 2, 3, 3]      3         4                 0
Average restarts: 2.7



Single-start hill climbing success rate: 40.3%
Random-restart hill climbing success rate: 100.0%


### Task 3: Local Beam Search for N-Queens

In [ ]:
def local_beam_search(k=3, max_iterations=100, verbose=True):
    current_states = [generate_initial_state() for _ in range(k)]

    for iteration in range(max_iterations):
        if verbose:
            print(f"Iteration {iteration}: beam = {[(st, calculate_heuristic(st)) for st in current_states]}")

        for state in current_states:
            if calculate_heuristic(state) == 0:
                return state, iteration

        all_successors = []
        for state in current_states:
            neighbors = generate_neighbors(state)
            all_successors.extend(neighbors)

        unique = [list(t) for t in set(map(tuple, all_successors))]
        unique.sort(key=calculate_heuristic)
        current_states = unique[:k]

    return None, max_iterations


for k in [1, 2, 3, 5]:
    print(f"\n===== k = {k} =====")
    solution, iterations = local_beam_search(k=k)
    print("Solution:", solution, "found at iteration:", iterations)
    if solution:
        print_board(solution)

print("\nSummary over 200 runs each")
for k in [1, 2, 3, 5]:
    results = [local_beam_search(k=k, verbose=False) for _ in range(200)]
    solved = [it for sol, it in results if sol]
    print(f"k={k}: success {len(solved) / 200:.1%}, avg iterations {sum(solved) / max(len(solved), 1):.2f}")


===== k = 1 =====
Iteration 0: beam = [([2, 0, 2, 1], 2)]
Iteration 1: beam = [([2, 0, 3, 1], 0)]
Solution: [2, 0, 3, 1] found at iteration: 1
- Q - -
- - - Q
Q - - -
- - Q -

===== k = 2 =====
Iteration 0: beam = [([0, 0, 1, 3], 3), ([2, 3, 2, 0], 3)]
Iteration 1: beam = [([1, 3, 2, 0], 1), ([2, 0, 1, 3], 1)]
Iteration 2: beam = [([2, 0, 3, 3], 1), ([1, 3, 0, 0], 1)]
Iteration 3: beam = [([2, 0, 3, 1], 0), ([1, 3, 0, 2], 0)]
Solution: [2, 0, 3, 1] found at iteration: 3
- Q - -
- - - Q
Q - - -
- - Q -

===== k = 3 =====
Iteration 0: beam = [([3, 0, 0, 3], 2), ([1, 0, 0, 2], 3), ([2, 2, 2, 2], 6)]
Iteration 1: beam = [([1, 3, 0, 2], 0), ([3, 0, 0, 2], 2), ([2, 0, 0, 3], 2)]
Solution: [1, 3, 0, 2] found at iteration: 1
- - Q -
Q - - -
- - - Q
- Q - -

===== k = 5 =====
Iteration 0: beam = [([2, 2, 1, 1], 3), ([2, 3, 1, 1], 3), ([3, 3, 1, 2], 3), ([3, 0, 2, 0], 2), ([1, 3, 3, 0], 2)]
Iteration 1: beam = [([1, 3, 2, 0], 1), ([3, 0, 2, 1], 1), ([0, 3, 1, 2], 1), ([3, 3, 0, 2], 1), ([1, 3, 

k=1: success 100.0%, avg iterations 2.94
k=2: success 100.0%, avg iterations 2.29
k=3: success 100.0%, avg iterations 1.90
k=5: success 100.0%, avg iterations 1.65


### Task 4: Beam Search on a Heuristic Graph

In [ ]:
graph = {
    "a": ["b", "j", "f"],
    "b": ["d", "c"],
    "c": ["h"],
    "j": ["k"],
    "f": ["e", "g"],
    "e": ["i"],
    "i": ["k"],
    "d": [],
    "g": [],
    "h": [],
    "k": [],
}


heuristic = {
    "a": 10, "b": 10, "c": 2, "d": 4, "e": 5, "f": 7,
    "g": 3, "h": 0, "i": 6, "j": 8, "k": 0,
}


def beam_search(start, goal, k=2):
    current_beam = [start]
    level = 0

    while current_beam:
        print(f"Level {level}: beam = {[(n, heuristic[n]) for n in current_beam]}")

        if goal in current_beam:
            print(f"Goal {goal} selected")
            return goal

        candidates = []
        for node in current_beam:
            children = graph[node]
            print(f"    {node} -> children {[(c, heuristic[c]) for c in children]}")
            candidates.extend(c for c in children if c not in candidates)

        candidates.sort(key=lambda n: heuristic[n])
        current_beam = candidates[:k]
        print(f"    Selected best {k}: {[(n, heuristic[n]) for n in current_beam]}")
        level += 1

    print("Goal not found")
    return None


for k in [2, 1, 3]:
    print(f"\n===== k = {k} =====")
    result = beam_search("a", "k", k=k)
    print("Result:", result)


===== k = 2 =====
Level 0: beam = [('a', 10)]
    a -> children [('b', 10), ('j', 8), ('f', 7)]
    Selected best 2: [('f', 7), ('j', 8)]
Level 1: beam = [('f', 7), ('j', 8)]
    f -> children [('e', 5), ('g', 3)]
    j -> children [('k', 0)]
    Selected best 2: [('k', 0), ('g', 3)]
Level 2: beam = [('k', 0), ('g', 3)]
Goal k selected
Result: k

===== k = 1 =====
Level 0: beam = [('a', 10)]
    a -> children [('b', 10), ('j', 8), ('f', 7)]
    Selected best 1: [('f', 7)]
Level 1: beam = [('f', 7)]
    f -> children [('e', 5), ('g', 3)]
    Selected best 1: [('g', 3)]
Level 2: beam = [('g', 3)]
    g -> children []
    Selected best 1: []
Goal not found
Result: None

===== k = 3 =====
Level 0: beam = [('a', 10)]
    a -> children [('b', 10), ('j', 8), ('f', 7)]
    Selected best 3: [('f', 7), ('j', 8), ('b', 10)]
Level 1: beam = [('f', 7), ('j', 8), ('b', 10)]
    f -> children [('e', 5), ('g', 3)]
    j -> children [('k', 0)]
    b -> children [('d', 4), ('c', 2)]
    Selected best 3

### Task 5: Simulated Annealing for N-Queens

In [ ]:
def simulated_annealing(
        initial_state,
        initial_temperature=100,
        cooling_rate=0.95,
        minimum_temperature=0.01,
        max_iterations=1000,
        verbose=True):

    current = initial_state
    temperature = initial_temperature

    for iteration in range(max_iterations):
        current_h = calculate_heuristic(current)

        if current_h == 0:
            if verbose:
                print(f"Iter {iteration}: solution found {current}")
            return current, iteration

        if temperature < minimum_temperature:
            break

        next_state = random.choice(generate_neighbors(current))
        next_h = calculate_heuristic(next_state)
        delta = next_h - current_h

        if delta < 0:
            accepted = True
        else:
            accepted = random.random() < math.exp(-delta / temperature)

        if verbose:
            print(f"Iter {iteration}: state = {current}, h = {current_h}, T = {temperature:.3f}, "
                  f"neighbor = {next_state}, h = {next_h}, {'accepted' if accepted else 'rejected'}")

        if accepted:
            current = next_state

        temperature *= cooling_rate

    return current, max_iterations

In [ ]:
random.seed(42)

initial_state = generate_initial_state()

print("Initial State:")
print_board(initial_state)

solution, _ = simulated_annealing(initial_state=initial_state)

print("\nFinal State:")
print_board(solution)
print("Final h:", calculate_heuristic(solution))

print("\nCooling rate comparison over 500 runs each")
for rate in [0.80, 0.95, 0.99]:
    results = [simulated_annealing(generate_initial_state(), cooling_rate=rate, verbose=False) for _ in range(500)]
    solved = [it for st, it in results if calculate_heuristic(st) == 0]
    print(f"cooling_rate={rate}: success {len(solved) / 500:.1%}, avg iterations to solve {sum(solved) / max(len(solved), 1):.1f}")

Initial State:
Q Q - -
- - - Q
- - Q -
- - - -
Iter 0: state = [0, 0, 2, 1], h = 3, T = 100.000, neighbor = [0, 1, 2, 1], h = 5, accepted
Iter 1: state = [0, 1, 2, 1], h = 5, T = 95.000, neighbor = [2, 1, 2, 1], h = 5, accepted
Iter 2: state = [2, 1, 2, 1], h = 5, T = 90.250, neighbor = [2, 1, 3, 1], h = 2, accepted
Iter 3: state = [2, 1, 3, 1], h = 2, T = 85.737, neighbor = [1, 1, 3, 1], h = 4, accepted
Iter 4: state = [1, 1, 3, 1], h = 4, T = 81.451, neighbor = [0, 1, 3, 1], h = 2, accepted
Iter 5: state = [0, 1, 3, 1], h = 2, T = 77.378, neighbor = [1, 1, 3, 1], h = 4, accepted
Iter 6: state = [1, 1, 3, 1], h = 4, T = 73.509, neighbor = [1, 0, 3, 1], h = 3, accepted
Iter 7: state = [1, 0, 3, 1], h = 3, T = 69.834, neighbor = [1, 0, 2, 1], h = 3, accepted
Iter 8: state = [1, 0, 2, 1], h = 3, T = 66.342, neighbor = [1, 0, 3, 1], h = 3, accepted
Iter 9: state = [1, 0, 3, 1], h = 3, T = 63.025, neighbor = [1, 0, 3, 2], h = 4, accepted
Iter 10: state = [1, 0, 3, 2], h = 4, T = 59.874, ne

cooling_rate=0.8: success 43.2%, avg iterations to solve 25.2


cooling_rate=0.95: success 95.4%, avg iterations to solve 74.8


cooling_rate=0.99: success 100.0%, avg iterations to solve 122.3
